**IMPROVEMENTS**

* [] pass the json for validation directly to the model invocation when possible (check if it's supported) i.e. stop to do the validation after the invocation

Self healing strategies:

* [x] increase temperature error after error (already implemented, works sometimes for qwen3)
* [ ] pass the error message when problems in the json are detected to see if it fixes them afterwards (new prompt with schema, invalid JSON, validation error and "Fix the JSON so it passes validation. Do not change values unless required to fix an error.")

In [3]:
import sys
import time
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os

from prompts import SYSTEM_PROMPT
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction, create_base_messages, invoke_model

In [4]:
LANGUAGE = "en" #TODO
MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
# MODEL_ID = "openai/gpt-oss-120b"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_dynamic-temp_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

FS_DATASET = "NLP-FBK/dyspnea-crf-train"
DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-development"
NUM_FS_EXAMPLES = 3
MAX_RETRIES = 3

API_KEY = "REDACTED_API_KEY" #TODO move
if not API_KEY: #TODO remove or make it useful
    print("Error: GROQ_API_KEY environment variable not set.")
    print("Please export your API key: export GROQ_API_KEY='your_key_here'")
    raise ValueError

In [5]:
os.chdir("..")

# one-time initializations
client = Groq(api_key=API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

base_messages = create_base_messages(CRFModel, SYSTEM_PROMPT, FS_DATASET, LANGUAGE, NUM_FS_EXAMPLES)
base_messages

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-development, loading from disk...
NLP-FBK/dyspnea-crf-development has 80 documents
Found local cache at data/dyspnea-crf-train, loading from disk...


[{'role': 'system',
  'content': "\nYou extract structured clinical data from a single clinical note.\n\nTask:\nPopulate a Dyspnea Case Report Form (CRF) using ONLY information explicitly stated in the note.\n\nRules:\n- Extract data strictly from the text; do not infer, guess, or normalize.\n- Populate a field ONLY if there is clear, explicit evidence.\n- If information is absent, unclear, ambiguous, or contradictory, treat it as unknown.\n- OMIT fields with unknown values from the output JSON.\n- All fields are strings.\n- For each populated field, output EXACTLY one of the allowed values defined in the schema.\n- Do not add, remove, rename fields, or invent values.\n- Output MUST be valid, minified JSON only (no whitespace, no explanations, no extra text).\n\n\nCRF Schema:\nchronic pulmonary disease: certainly chronic | possibly chronic | certainly not chronic | unknown\nchronic respiratory failure: certainly chronic | possibly chronic | certainly not chronic | unknown\nchronic card

In [ ]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

In [8]:
#TODO clean this functiton
def crf_dataset():
    with open(OUTPUT_PATH, "a") as f: #TODO this is append now
        for item in ds:
            doc_id = item["document_id"]
            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            clinical_note = item["clinical_note"]
            
            print(f"Processing document: {doc_id}")
            success = False

            for attempt in range(MAX_RETRIES):
                try:
                    # 1. Invoke model (Network/API errors cached here)
                    raw_output = invoke_model(client, MODEL_ID, clinical_note, base_messages.copy(), attempt)
                    
                    # 2. Parse JSON
                    data_dict = json.loads(raw_output)
                    
                    # 3. Validate with Pydantic
                    result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success

                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except Exception as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts.")
                
                # Fallback: Create fake instance with all "unknown"
                fake_instance = CRFModel()
                save_prediction(f, doc_id, fake_instance)

crf_dataset()

Skipped 1014081_en because already processed
Skipped 1117595_en because already processed
Skipped 114284_en because already processed
Skipped 1147182_en because already processed
Skipped 1166267_en because already processed
Skipped 1231372_en because already processed
Skipped 1236585_en because already processed
Skipped 1258026_en because already processed
Skipped 1307609_en because already processed
Skipped 1323348_en because already processed
Skipped 1323973_en because already processed
Skipped 1350262_en because already processed
Skipped 1486650_en because already processed
Skipped 1498690_en because already processed
Skipped 1526466_en because already processed
Skipped 15455_en because already processed
Skipped 1565152_en because already processed
Skipped 1706077_en because already processed
Skipped 1769015_en because already processed
Skipped 1777967_en because already processed
Skipped 1803820_en because already processed
Skipped 1833659_en because already processed
Skipped 24383